In [ ]:
import cv2
import numpy as np

# Path to the image
image_path = r"C:\Users\nishi\Downloads\cpen 391 srcipts\SignSafety\jupyter_scripts\speed.png"

# Step 3: Read the uploaded image
original_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Function to update parameters
def update_parameters():
    # Set parameter values directly
    blur_kernel = 12  # Ensure odd value
    canny_low = 0
    canny_high = 195
    threshold_value = 127
    scale = 30  # Resize percentage (100 means no resizing)

    # Ensure blur_kernel is odd
    blur_kernel = max(3, blur_kernel | 1)  # Make it odd and at least 3

    # Resize scale factor
    resize_factor = max(scale / 100, 0.01)

    # Apply Gaussian Blur
    blurred_image = cv2.GaussianBlur(original_image, (blur_kernel, blur_kernel), 1.4)

    # Apply Canny edge detection
    edges = cv2.Canny(blurred_image, canny_low, canny_high)

    # Apply threshold
    _, threshold = cv2.threshold(edges, threshold_value, 255, cv2.THRESH_BINARY)

    # Find contours
    contours, _ = cv2.findContours(threshold, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Create a color copy of the image for highlighting
    image_with_highlight = cv2.cvtColor(original_image, cv2.COLOR_GRAY2BGR)

    for i, contour in enumerate(contours):
        # Skip the first contour which is the whole image
        if i == 0:
            continue

        # Approximate the contour
        approx = cv2.approxPolyDP(contour, 0.01 * cv2.arcLength(contour, True), True)

        # Highlight quadrilaterals
        if len(approx) == 4:  # Check if the contour has 4 vertices
            cv2.polylines(image_with_highlight, [approx], isClosed=True, color=(0, 255, 0), thickness=3)  # Green outline
            cv2.fillPoly(image_with_highlight, [approx], color=(0, 255, 0))  # Fill with translucent color (optional)

            # Calculate the center of the shape
            M = cv2.moments(contour)
            if M['m00'] != 0.0:
                x = int(M['m10'] / M['m00'])
                y = int(M['m01'] / M['m00'])

                # Add a label at the center of the shape
                cv2.putText(image_with_highlight, 'Quadrilateral', (x - 40, y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Resize the image
    resized_original = cv2.resize(original_image, None, fx=resize_factor, fy=resize_factor, interpolation=cv2.INTER_AREA)
    resized_processed = cv2.resize(image_with_highlight, None, fx=resize_factor, fy=resize_factor, interpolation=cv2.INTER_AREA)

    # Convert original grayscale image to BGR for side-by-side display
    resized_original = cv2.cvtColor(resized_original, cv2.COLOR_GRAY2BGR)

    # Concatenate the images horizontally
    combined_image = np.hstack((resized_original, resized_processed))

    # Show the combined image
    cv2.imshow('Original and Processed Images', combined_image)

# Call the function to process and display the image
update_parameters()

# Wait until a key is pressed
cv2.waitKey(0)
cv2.destroyAllWindows()


In [18]:
import cv2
import numpy as np

# Use a raw string to avoid escape character issues
image_path = r"C:\Users\nishi\Downloads\cpen 391 srcipts\SignSafety\jupyter_scripts\rgb_image.png"
image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

# Check if the image was loaded successfully
if image is None:
    print(f"Error: Unable to load image at {image_path}")
else:
    # Convert the image to HSV color space
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    
    # Define range for red color in HSV
    lower_red1 = np.array([0, 100, 60])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 100, 50])
    upper_red2 = np.array([180, 255, 255])
    
    # Create masks for red color ranges
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    red_mask = cv2.bitwise_or(mask1, mask2)
    
    # Apply the mask to the original image
    red_highlighted = cv2.bitwise_and(image, image, mask=red_mask)
    
    # Optional: Find contours and draw bounding boxes around red areas
    contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w > 10 and h > 10:  # Filter out small regions
            cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)  # Green bounding box
    
    # Display the original image with bounding boxes
    cv2.imshow("Detected Red Regions", image)
    
    # Display the red-highlighted image
    cv2.imshow("Red Highlighted", red_highlighted)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
